In [114]:
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.optim as optim


import matplotlib.pyplot as plt

In [115]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/DL/CIFAR100'
    print("✅ Contected to Google Drive")
except(ValueError, ImportError, Exception) as e:
    print(f"❌ Error: {e}")
    print("❌ Failed to mount to Google Drive --> ✅ Use Local Path")
    DATA_PATH = "./data"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Contected to Google Drive


In [116]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [117]:
IMG_SIZE = 32
PATCH_SIZE = 4
IN_CHANNELS = 3
BATCH_SIZE = 128

# Training
LR = 3e-4
DROP_RATE = 0.4
EPOCHS = 200

# ViT
EMBED_DIM = 256
MLP_DIM = 512
ENC_NUMS = 6
MSA_NUMS = 8
CLS_NUMS = 100

In [118]:
stats = ((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))

train_transform = transforms.Compose([
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize(*stats),
    transforms.RandomErasing(p=0.25)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

In [119]:
from torch.utils.data import random_split

full_data = datasets.CIFAR100(root=DATA_PATH, train=True, download=True, transform=train_transform)

train_size = int(0.8 * len(full_data))
val_size = len(full_data) - train_size

train_subset, val_subset = random_split(full_data, [train_size, val_size])

test_data = datasets.CIFAR100(root=DATA_PATH, train=False, download=True, transform=test_transform)

In [120]:
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=4, pin_memory=True)

val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=4, pin_memory=True)
                            
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=4, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [121]:
class EmbeddedPatches(nn.Module):
    def __init__(self, img_size, in_channels, embed_dim, patch_size, batch_size):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, patch_size, patch_size)
        N = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_token = nn.Parameter(torch.randn(1, N+1, embed_dim))

    def forward(self, x):
        B = x.size(0)
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        cls_token = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.pos_token

        return x

In [122]:
class MLP(nn.Module):
    def __init__(self, in_features, out_features, drop_rate):
        super().__init__()
        self.layer1 = nn.Linear(in_features, out_features)
        self.layer2 = nn.Linear(out_features, in_features)
        self.dropout = nn.Dropout(drop_rate)

    def forward(self, x):
        x = self.layer1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.layer2(x)
        x = self.dropout(x)

        return x

In [123]:
class VisionEncoder(nn.Module):
    def __init__(self, embed_dim, msa_size, mlp_dim, enc_dim, drop_rate):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, msa_size, drop_rate, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, mlp_dim, drop_rate)

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))

        return x


In [124]:
class ViT(nn.Module):
    def __init__(self, img_size, patch_size, batch_size, in_channels, embed_dim, enc_dim, msa_size, mlp_dim, cls_nums, drop_rate):
        super().__init__()
        self.embed = EmbeddedPatches(img_size, in_channels, embed_dim, patch_size, batch_size)
        self.encoder = nn.Sequential(*[
            VisionEncoder(embed_dim, msa_size, mlp_dim, enc_dim, drop_rate)
            for _ in range(enc_dim)
        ])
        self.mlp_head = nn.Linear(embed_dim, cls_nums)

    def forward(self, x):
        x = self.embed(x)
        x = self.encoder(x)
        cls_token = x[:, 0]
        x = self.mlp_head(cls_token)

        return x

In [125]:
model = ViT(IMG_SIZE, PATCH_SIZE, BATCH_SIZE, IN_CHANNELS, 
            EMBED_DIM, ENC_NUMS, MSA_NUMS, MLP_DIM, CLS_NUMS, DROP_RATE)
            
model.to(device)

ViT(
  (embed): EmbeddedPatches(
    (proj): Conv2d(3, 256, kernel_size=(4, 4), stride=(4, 4))
  )
  (encoder): Sequential(
    (0): VisionEncoder(
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (layer1): Linear(in_features=256, out_features=512, bias=True)
        (layer2): Linear(in_features=512, out_features=256, bias=True)
        (dropout): Dropout(p=0.4, inplace=False)
      )
    )
    (1): VisionEncoder(
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (layer1): Linear(in_fea

In [126]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.1,
    div_factor=10,
    final_div_factor=100
)

In [127]:
import numpy as np

class EarlyStopping:
    """Early stops the training if validation loss doesn't improve after a given patience."""
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pth', trace_func=print):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.
                            Default: 7
            verbose (bool): If True, prints a message for each validation loss improvement. 
                            Default: False
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
                            Default: 0
            path (str): Path for the checkpoint to be saved to.
                            Default: 'checkpoint.pth'
            trace_func (function): trace print function.
                            Default: print            
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func
    
    def __call__(self, val_loss, model):

        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        '''Saves model when validation loss decrease.'''
        if self.verbose:
            self.trace_func(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

In [128]:
def train(model, loader, optimizer, criterion, scheduler):
    model.train()
    total_loss, correct = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)

In [129]:
def validate(model, loader, criterion):
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            val_loss += loss.item() * x.size(0)

    return val_loss / len(loader.dataset)

In [130]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(1) == y).sum().item()
    
    return correct / len(loader.dataset)

In [ ]:
train_accuracies, test_accuracies = [], []
early_stopping = EarlyStopping(patience=10, path='best_model.pth')

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, scheduler)
    
    val_loss = validate(model, val_loader, criterion)
    
    test_acc = evaluate(model, test_loader)

    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    print(f"Epoch: {epoch}, train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f}, val_loss: {val_loss:.4f}, test_acc: {test_acc:.4f}")

    early_stopping(val_loss, model)

    if early_stopping.early_stop:
        print("Early Stopping")
        break

model.load_state_dict(torch.load('best_model.pth'))
print("Loaded best model")

Epoch: 0, train_loss: 4.6184, train_acc: 0.0201, val_loss: 4.4766, test_acc: 0.0500
Epoch: 1, train_loss: 4.4718, train_acc: 0.0357, val_loss: 4.4035, test_acc: 0.0671
Epoch: 2, train_loss: 4.3925, train_acc: 0.0486, val_loss: 4.3227, test_acc: 0.0987
Epoch: 3, train_loss: 4.3111, train_acc: 0.0631, val_loss: 4.2465, test_acc: 0.1166
Epoch: 4, train_loss: 4.2376, train_acc: 0.0762, val_loss: 4.1741, test_acc: 0.1414
Epoch: 5, train_loss: 4.1638, train_acc: 0.0909, val_loss: 4.1009, test_acc: 0.1609
Epoch: 6, train_loss: 4.1008, train_acc: 0.1027, val_loss: 4.0374, test_acc: 0.1827
Epoch: 7, train_loss: 4.0406, train_acc: 0.1158, val_loss: 3.9666, test_acc: 0.1999
Epoch: 8, train_loss: 3.9729, train_acc: 0.1263, val_loss: 3.9285, test_acc: 0.2076
Epoch: 9, train_loss: 3.9312, train_acc: 0.1325, val_loss: 3.8736, test_acc: 0.2321
Epoch: 10, train_loss: 3.8825, train_acc: 0.1442, val_loss: 3.8314, test_acc: 0.2440
Epoch: 11, train_loss: 3.8476, train_acc: 0.1512, val_loss: 3.7828, test_ac

In [ ]:
import os

model_name = 'pre_trained_ViT_CIFAR100.pth'
save_path = os.path.join("/content/drive/MyDrive/DL", model_name)

torch.save(model.state_dict(), save_path)

print(f"✅ Model saved to: {save_path}")

In [ ]:
plt.plot(train_accuracies, label='Train Accuracy')
plt.plot(test_accuracies, label='Test Accuracy')
plt.ylabel("Accuracy")
plt.xlabel("Epochs")
plt.legend()
plt.title("Train and Test Accuracy")
plt.show()

In [ ]:
# --- Visualisierung: Attention Rollout (Abnar & Zuidema, 2020) ---
import numpy as np

# 1. Speicher für die Attention Weights
attention_maps = []

# 2. Hook-Funktion: Greift die Daten ab, wenn sie durch den Layer fließen
def hook_fn(module, input, output):
    # output bei MultiheadAttention ist (attn_output, attn_weights)
    # attn_weights shape (Standard): (Batch, Seq_Len, Seq_Len) 
    # (PyTorch mittelt die Heads standardmäßig bereits)
    attention_maps.append(output[1].detach().cpu())

def visualize_attention_rollout(model, loader, num_images=5):
    model.eval()
    
    # --- A. Hooks registrieren ---
    # Wir hängen uns an jedes 'attn' Modul in deinem TransformerEncoderWrapper
    # Hinweis: Dein Encoder ist ein nn.Sequential. 
    handles = []
    for layer in model.encoder:
        # layer ist hier ein TransformerEncoderLayer
        handle = layer.attn.register_forward_hook(hook_fn)
        handles.append(handle)
        
    # --- B. Bilder durch das Modell schicken ---
    # Hole einen Batch Bilder aus dem Loader
    try:
        images, labels = next(iter(loader))
    except StopIteration:
        return
        
    images = images[:num_images].to(device)
    labels = labels[:num_images]
    
    global attention_maps
    attention_maps = [] # Liste leeren
    
    with torch.no_grad():
        output = model(images) # Dies füllt attention_maps!
        
    # --- C. Attention Rollout berechnen ---
    rollout_maps = []
    
    for i in range(num_images):
        # Seq_Len ist Anzahl Patches + 1 (CLS Token)
        # Wir nehmen die Matrix von Layer 0 um Dimensionen zu prüfen
        seq_len = attention_maps[0].shape[2] 
        
        # Start mit Identitätsmatrix (für die Residual Connection "durchgereicht")
        result = torch.eye(seq_len)
        
        for layer_attn in attention_maps:
            # layer_attn Shape: (Batch, Seq, Seq) -> wir nehmen Bild i
            attn = layer_attn[i] # (Seq, Seq)
            
            # Abnar & Zuidema: Residual Connection modellieren
            # Formel: A_new = 0.5 * A + 0.5 * Identity
            # Das simuliert, dass Information sowohl durch Attention als auch 
            # durch den Residual Skip (x + attn(x)) fließt.
            identity = torch.eye(seq_len)
            a_hat = 0.5 * attn + 0.5 * identity
            
            # Renormalisieren (Zeilensumme = 1)
            a_hat = a_hat / a_hat.sum(dim=-1, keepdim=True)
            
            # Matrix Multiplikation: Der Fluss der Information von unten nach oben
            result = torch.matmul(a_hat, result)
            
        # Wir schauen uns die Zeile 0 an: Wie viel Aufmerksamkeit floss zum [CLS] Token (Index 0)?
        # Wir ignorieren Index 0 selbst und nehmen die Patches 1 bis Ende
        mask = result[0, 1:]
        
        # Reshape zurück in 2D Grid (z.B. 8x8 bei 32x32 Bild und Patch 4)
        width = int(np.sqrt(len(mask)))
        mask = mask.reshape(width, width)
        
        # Upscale auf Bildgröße (32x32) mittels PyTorch Interpolate
        # Wir müssen Dimensionen unsequeezing: (1, 1, H, W)
        mask = mask.unsqueeze(0).unsqueeze(0)
        mask = F.interpolate(mask, size=(img_size, img_size), mode='bilinear', align_corners=False)
        mask = mask.squeeze().numpy()
        
        # Normalisieren für Plot (0 bis 1)
        mask = (mask - mask.min()) / (mask.max() - mask.min())
        rollout_maps.append(mask)

    # --- D. Plotten ---
    # Hooks entfernen
    for handle in handles:
        handle.remove()
        
    # Plot erstellen
    fig, axes = plt.subplots(2, num_images, figsize=(3 * num_images, 6))
    
    for i in range(num_images):
        # Originalbild de-normalisieren für Anzeige
        img = images[i].cpu().permute(1, 2, 0).numpy()
        # CIFAR10 Stats (Mean, Std) rückgängig machen
        mean = np.array(stats[0])
        std = np.array(stats[1])
        img = std * img + mean
        img = np.clip(img, 0, 1)
        
        # Zeile 1: Originalbild
        axes[0, i].imshow(img)
        axes[0, i].set_title(f"Class: {labels[i].item()}")
        axes[0, i].axis('off')
        
        # Zeile 2: Attention Overlay
        axes[1, i].imshow(img)
        # 'jet' ist eine gute Heatmap, alpha mischt es mit dem Bild
        axes[1, i].imshow(rollout_maps[i], alpha=0.6, cmap='jet') 
        axes[1, i].set_title("Attention Flow")
        axes[1, i].axis('off')
        
    plt.tight_layout()
    plt.show()

# --- Funktionsaufruf ---
# Wir benötigen 'img_size' für das Upscaling, das war oben definiert als IMAGE_SIZE
img_size = IMG_SIZE 
visualize_attention_rollout(model, test_loader)